## Create dataframe + extract date parts

In [2]:
import pandas as pd

views = pd.read_csv(
    "../data/feed-views.log",
    sep="\t",
    header=None,
    names=["datetime", "user"]
)

views["datetime"] = pd.to_datetime(views["datetime"])

views["year"] = views["datetime"].dt.year
views["month"] = views["datetime"].dt.month
views["day"] = views["datetime"].dt.day
views["hour"] = views["datetime"].dt.hour
views["minute"] = views["datetime"].dt.minute
views["second"] = views["datetime"].dt.second


## Create daytime using cut

In [3]:
bins = [0, 4, 7, 11, 17, 20, 24]
labels = [
    "night",
    "early morning",
    "morning",
    "afternoon",
    "early evening",
    "evening"
]

views["daytime"] = pd.cut(
    views["hour"],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

views = views.set_index("user")

## Count

In [4]:
views.count()
views["daytime"].value_counts()

daytime
evening          509
afternoon        252
early evening    145
night            129
morning           36
early morning      5
Name: count, dtype: int64

## Sort by hour, minute, second

In [5]:
views = views.sort_values(["hour", "minute", "second"])

## Min/max hours + mode + examples

In [6]:
min_hour = views["hour"].min()
max_hour = views["hour"].max()

max_night_hour = views.loc[views["daytime"] == "night", "hour"].max()
min_morning_hour = views.loc[views["daytime"] == "morning", "hour"].min()

night_example = views.loc[views["hour"] == max_night_hour].head(1)
morning_example = views.loc[views["hour"] == min_morning_hour].head(1)

mode_hour = views["hour"].mode().iloc[0]
mode_daytime = views["daytime"].mode().iloc[0]

print(f"Min hour: {min_hour}")
print(f"Max hour: {max_hour}")
print(f"Max night hour: {max_night_hour}")
print(f"Min morning hour: {min_morning_hour}")
print(f"Mode hour: {mode_hour}")
print(f"Mode daytime: {mode_daytime}")

print("\nNight example:")
display(night_example)

print("\nMorning example:")
display(morning_example)

Min hour: 0
Max hour: 23
Max night hour: 3
Min morning hour: 8
Mode hour: 22
Mode daytime: evening

Night example:


,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
konstantin,2020-04-19 03:23:35.471598,2020,4,19,3,23,35,night



Morning example:


,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
alexander,2020-05-15 08:16:03.918402,2020,5,15,8,16,3,morning


## 3 earliest and latest hours + usernames

In [7]:
earliest = views.nsmallest(3, "hour")[["datetime", "hour"]]
latest = views.nlargest(3, "hour")[["datetime", "hour"]]

print("3 earliest visits:")
display(earliest)

print("\n3 latest visits:")
display(latest)

3 earliest visits:


,datetime,hour
user,,
valentina,2020-05-15 00:00:13.222265,0
valentina,2020-05-15 00:01:05.153738,0
pavel,2020-05-12 00:01:27.764025,0



3 latest visits:


,datetime,hour
user,,
ekaterina,2020-05-14 23:02:11.327532,23
ekaterina,2020-05-14 23:02:14.494985,23
ekaterina,2020-05-14 23:02:15.588808,23


## Describe + IQR for hour

In [8]:
desc = views.describe()
iqr = desc.loc["75%", "hour"] - desc.loc["25%", "hour"]
display(desc)

print(f"IQR for hour: {iqr}")

,datetime,year,month,day,hour,minute,second
count,1076,1076.0,1076.000000,1076.000000,1076.000000,1076.000000,1076.000000
mean,2020-05-10 09:00:41.211420672,2020.0,4.870818,13.552974,16.249071,29.629182,29.500929
min,2020-04-17 12:01:08.463179,2020.0,4.000000,1.000000,0.000000,0.000000,0.000000
25%,2020-05-10 01:13:49.857472,2020.0,5.000000,11.000000,13.000000,14.000000,14.000000
50%,2020-05-11 22:48:35.302552832,2020.0,5.000000,13.000000,19.000000,29.000000,30.000000
75%,2020-05-14 14:44:34.749530624,2020.0,5.000000,15.000000,22.000000,46.000000,45.000000
max,2020-05-22 10:36:14.662600,2020.0,5.000000,30.000000,23.000000,59.000000,59.000000
std,NaN,0.0,0.335557,4.906567,6.955490,17.689388,17.405506


IQR for hour: 9.0
